# MLflow en Databricks: clasificación de tumores de mama

**Autor:** Franco · Seminario de Experiencias en Analítica, EAFIT

**Objetivo:** reproducir el flujo completo de MLflow en Databricks con un dataset distinto al de clase:
registrar experimentos, comparar modelos, versionarlos en Unity Catalog y consumirlos por alias.

**Caso:** clasificar tumores de mama como **malignos (0)** o **benignos (1)** a partir de 30 medidas
calculadas sobre imágenes de biopsias (dataset `breast_cancer` de scikit-learn, 569 muestras).
Se descarga con la librería, no hay que cargar archivos.

### Diferencias frente al notebook de clase

| Tema | Notebook de clase (vinos) | Este notebook |
|---|---|---|
| Problema | Multiclase (3 variedades) | Binario (maligno / benigno) |
| Tamaño | 178 × 13 | 569 × 30 |
| Métricas | accuracy, f1_macro | accuracy, f1, **recall_maligno**, **roc_auc** |
| Selección de hiperparámetros | Sobre el set de prueba | Sobre un set de **validación**; el de prueba se usa una sola vez |

En este problema el error grave es decir "benigno" cuando el tumor es maligno, por eso además del F1
registramos el **recall de la clase maligna**.

| Concepto MLflow | Ejemplo aquí |
|---|---|
| Experimento | `curso-mlflow-cancer-mama` |
| Run | `regresion_logistica` |
| Parámetro | `C = 1.0` |
| Métrica | `roc_auc = 0.99` |
| Artefacto | matriz de confusión, curva ROC |
| Modelo registrado | `workspace.default.clasificador_cancer_mama` |
| Alias | `@champion`, `@challenger` |

## 0. Configuración
Instalamos una versión reciente de MLflow y reiniciamos Python.

In [ ]:
%pip install -q -U "mlflow>=3.1" "scikit-learn>=1.3" matplotlib

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("esquema", "default")

CATALOGO = dbutils.widgets.get("catalogo")
ESQUEMA = dbutils.widgets.get("esquema")
NOMBRE_MODELO = f"{CATALOGO}.{ESQUEMA}.clasificador_cancer_mama"

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from mlflow import MlflowClient
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, f1_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Los modelos registrados viven en Unity Catalog.
mlflow.set_registry_uri("databricks-uc")

# El experimento se crea en la carpeta del usuario si no existe.
usuario = spark.sql("SELECT current_user()").first()[0]
experimento = mlflow.set_experiment(f"/Users/{usuario}/curso-mlflow-cancer-mama")

# Tipos internos de scikit-learn que MLflow (formato skops) debe aceptar al cargar los modelos.
# Se declaran porque los modelos los entrenamos nosotros: nunca se confía en un modelo ajeno.
TIPOS_CONFIABLES = [
    "sklearn.tree._tree.Tree",
    "sklearn._loss.loss.HalfBinomialLoss",
    "sklearn._loss.link.LogitLink",
    "sklearn._loss.link.Interval",
]

print(f"[diag] Usuario       : {usuario}")
print(f"[diag] Experimento   : {experimento.name} (id {experimento.experiment_id})")
print(f"[diag] Modelo UC     : {NOMBRE_MODELO}")
print(f"[diag] MLflow        : {mlflow.__version__}")

### Datos: entrenamiento, validación y prueba
- **Entrenamiento (60 %)**: ajustar los modelos.
- **Validación (20 %)**: comparar configuraciones y elegir hiperparámetros.
- **Prueba (20 %)**: medir una sola vez el modelo final y comparar champion vs challenger.

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"[diag] Dataset completo : {X.shape}")
print(f"[diag] Entrenamiento    : {X_train.shape}")
print(f"[diag] Validación       : {X_val.shape}")
print(f"[diag] Prueba           : {X_test.shape}")
print("[diag] Proporción de clases (0=maligno, 1=benigno):")
print(y.value_counts(normalize=True).rename({0: "maligno", 1: "benigno"}).round(3).to_string())
display(X_train.head())

## 1. Primer run: línea base
Función auxiliar que calcula todas las métricas de un modelo binario, y un run con regresión logística.

In [ ]:
def metricas(modelo, X_eval, y_eval):
    """Devuelve un dict con las métricas del modelo sobre (X_eval, y_eval)."""
    pred = modelo.predict(X_eval)
    proba = modelo.predict_proba(X_eval)[:, 1]
    return {
        "accuracy": accuracy_score(y_eval, pred),
        "f1": f1_score(y_eval, pred),
        "recall_maligno": recall_score(y_eval, pred, pos_label=0),
        "roc_auc": roc_auc_score(y_eval, proba),
    }


def matriz_confusion(modelo, X_eval, y_eval, titulo):
    figura, eje = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay.from_estimator(
        modelo, X_eval, y_eval, display_labels=["maligno", "benigno"], ax=eje, colorbar=False
    )
    eje.set_title(titulo)
    plt.close(figura)
    return figura


def curva_roc(modelo, X_eval, y_eval, titulo):
    figura, eje = plt.subplots(figsize=(4, 4))
    RocCurveDisplay.from_estimator(modelo, X_eval, y_eval, ax=eje)
    eje.set_title(titulo)
    plt.close(figura)
    return figura


with mlflow.start_run(run_name="regresion_logistica") as run:
    C = 1.0
    modelo = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    modelo.fit(X_train, y_train)
    m = metricas(modelo, X_val, y_val)

    mlflow.log_params({"algoritmo": "LogisticRegression", "C": C})
    mlflow.log_metrics(m)
    mlflow.set_tags({"etapa": "linea_base", "autor": "Franco", "dataset": "breast_cancer"})
    mlflow.log_figure(matriz_confusion(modelo, X_val, y_val, "Regresión logística"), "matriz_confusion.png")
    mlflow.log_figure(curva_roc(modelo, X_val, y_val, "ROC - Regresión logística"), "curva_roc.png")
    mlflow.log_text("Línea base: modelo lineal con datos estandarizados.", "notas.txt")

print(f"[diag] Run ID: {run.info.run_id}")
for k, v in m.items():
    print(f"[diag] {k:<15}: {v:.4f}")

👉 **En la interfaz:** **Experiments › curso-mlflow-cancer-mama › regresion_logistica**.
Revisa *Overview* (parámetros, métricas, tags) y *Artifacts* (matriz, curva ROC, notas).

### Ejercicio 1: efecto de la regularización
Varios runs con distintos valores de `C`. Una C pequeña regulariza más.

In [ ]:
for C in [0.01, 0.1, 10.0]:
    with mlflow.start_run(run_name=f"regresion_logistica_C{C}") as run:
        modelo = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        modelo.fit(X_train, y_train)
        m = metricas(modelo, X_val, y_val)

        mlflow.log_params({"algoritmo": "LogisticRegression", "C": C})
        mlflow.log_metrics(m)
        mlflow.set_tags({"etapa": "ajuste_regularizacion", "autor": "Franco"})
        mlflow.log_figure(matriz_confusion(modelo, X_val, y_val, f"Regresión logística C={C}"),
                          "matriz_confusion.png")

    print(f"[diag] C={C:<6} f1={m['f1']:.4f}  recall_maligno={m['recall_maligno']:.4f}  "
          f"roc_auc={m['roc_auc']:.4f}  run={run.info.run_id[:8]}")

## 2. Métricas que cambian en el tiempo
Con `step` una métrica guarda una serie. Medimos el F1 de validación después de cada árbol del
*gradient boosting*.

In [ ]:
with mlflow.start_run(run_name="gradient_boosting_curva"):
    modelo = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42)
    modelo.fit(X_train, y_train)
    mlflow.log_params({"algoritmo": "GradientBoosting", "n_estimators": 200, "learning_rate": 0.1})

    curva = []
    for paso, pred_parcial in enumerate(modelo.staged_predict(X_val), start=1):
        f1_paso = f1_score(y_val, pred_parcial)
        curva.append(f1_paso)
        mlflow.log_metric("f1_validacion", f1_paso, step=paso)

    mlflow.log_metrics(metricas(modelo, X_val, y_val))

mejor_paso = max(range(len(curva)), key=lambda i: curva[i]) + 1
print(f"[diag] Pasos registrados : {len(curva)}")
print(f"[diag] Mejor F1 validación: {max(curva):.4f} en el árbol {mejor_paso}")
print(f"[diag] F1 final (árbol 200): {curva[-1]:.4f}")

👉 **En la interfaz:** en `gradient_boosting_curva`, pestaña *Model metrics*. ¿Desde qué árbol se
estabiliza el F1?

## 3. Búsqueda de hiperparámetros con runs anidados
Un run padre agrupa la búsqueda; cada combinación es un run hijo. Todo se evalúa en **validación**.

In [ ]:
combinaciones = [
    {"n_estimators": n, "max_depth": d, "min_samples_leaf": h}
    for n in [10, 50, 150]
    for d in [3, 6, None]
    for h in [1, 5]
]
print(f"[diag] Combinaciones a evaluar: {len(combinaciones)}")

resultados = []
with mlflow.start_run(run_name="busqueda_random_forest") as padre:
    mlflow.set_tags({"etapa": "busqueda", "autor": "Franco"})
    for params in combinaciones:
        nombre = f"rf_{params['n_estimators']}_prof_{params['max_depth']}_hoja_{params['min_samples_leaf']}"
        with mlflow.start_run(run_name=nombre, nested=True):
            modelo = RandomForestClassifier(random_state=42, **params).fit(X_train, y_train)
            m = metricas(modelo, X_val, y_val)
            mlflow.log_params({"algoritmo": "RandomForest", **params})
            mlflow.log_metrics(m)
            resultados.append({"run": nombre, **params, **m})
            print(f"  {nombre:<26} f1={m['f1']:.4f}  recall_mal={m['recall_maligno']:.4f}  "
                  f"auc={m['roc_auc']:.4f}")

    mejor_hijo = max(resultados, key=lambda r: (r["f1"], r["roc_auc"]))
    mlflow.log_metric("mejor_f1", mejor_hijo["f1"])
    mlflow.set_tag("mejor_run", mejor_hijo["run"])

print(f"\n[diag] Padre run ID: {padre.info.run_id}")
print(f"[diag] Mejor hijo  : {mejor_hijo['run']} (f1={mejor_hijo['f1']:.4f})")

👉 **En la interfaz:** expande `busqueda_random_forest`, selecciona los hijos y usa **Compare** con la
vista de *coordenadas paralelas* (`n_estimators`, `max_depth`, `min_samples_leaf`, `f1`).

## 4. Autolog
`mlflow.sklearn.autolog()` registra solo hiperparámetros, métricas de entrenamiento y gráficos.

In [ ]:
mlflow.sklearn.autolog(log_models=False)

with mlflow.start_run(run_name="autolog_random_forest") as run:
    modelo = RandomForestClassifier(n_estimators=80, max_depth=6, random_state=42)
    modelo.fit(X_train, y_train)
    mlflow.log_metrics({f"val_{k}": v for k, v in metricas(modelo, X_val, y_val).items()})

mlflow.sklearn.autolog(disable=True)

n_params = len(mlflow.get_run(run.info.run_id).data.params)
print(f"[diag] Parámetros registrados automáticamente: {n_params}")

👉 **En la interfaz:** compara cuántos parámetros registró `autolog_random_forest` contra los runs manuales.

## 5. Consultar runs desde código
`mlflow.search_runs` devuelve un DataFrame. Los parámetros llegan como **texto**, así que para desempatar
por simplicidad (menos árboles, menos profundidad) los convertimos a números antes de ordenar.

In [ ]:
runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    filter_string="params.algoritmo = 'RandomForest'",
)
runs = runs[runs["metrics.f1"].notna() & runs["params.min_samples_leaf"].notna()].copy()

runs["n_est"] = runs["params.n_estimators"].astype(int)
runs["prof"] = runs["params.max_depth"].replace("None", "999").astype(int)
runs["hoja"] = runs["params.min_samples_leaf"].astype(int)

# Mejor F1; empate -> mejor AUC; empate -> modelo más simple.
runs = runs.sort_values(
    ["metrics.f1", "metrics.roc_auc", "n_est", "prof"], ascending=[False, False, True, True]
)
columnas = ["tags.mlflow.runName", "n_est", "prof", "hoja", "metrics.f1", "metrics.roc_auc"]
display(runs[columnas].head(5))

mejor = runs.iloc[0]
mejores_params = {
    "n_estimators": int(mejor["n_est"]),
    "max_depth": None if mejor["params.max_depth"] == "None" else int(mejor["params.max_depth"]),
    "min_samples_leaf": int(mejor["hoja"]),
}
print(f"[diag] Runs candidatos   : {len(runs)}")
print(f"[diag] Mejor configuración: {mejores_params}")
print(f"[diag] F1 validación     : {mejor['metrics.f1']:.4f}")

## 6. Modelo final
Se entrena con la mejor configuración y **ahora sí** se mide en el set de prueba, una sola vez.
El modelo se guarda con firma (inferida del `input_example`) y dependencias.

In [ ]:
with mlflow.start_run(run_name="random_forest_final") as run_final:
    modelo_final = RandomForestClassifier(random_state=42, **mejores_params).fit(X_train, y_train)
    m_test = metricas(modelo_final, X_test, y_test)

    mlflow.log_params({"algoritmo": "RandomForest", **mejores_params})
    mlflow.log_metrics({f"test_{k}": v for k, v in m_test.items()})
    mlflow.set_tags({"etapa": "final", "autor": "Franco"})
    mlflow.log_figure(matriz_confusion(modelo_final, X_test, y_test, "RF final (prueba)"), "matriz_confusion.png")
    mlflow.log_figure(curva_roc(modelo_final, X_test, y_test, "ROC RF final (prueba)"), "curva_roc.png")
    info_rf = mlflow.sklearn.log_model(
        modelo_final,
        name="modelo",
        input_example=X_train.head(3),
        skops_trusted_types=TIPOS_CONFIABLES,
    )

print(f"[diag] Modelo guardado en: {info_rf.model_uri}")
for k, v in m_test.items():
    print(f"[diag] test_{k:<15}: {v:.4f}")
print(f"[diag] Columnas en la firma: {len(info_rf.signature.inputs.input_names())}")

In [ ]:
cargado = mlflow.pyfunc.load_model(info_rf.model_uri)
pred_cargado = cargado.predict(X_test.head(5))
print(f"[diag] Predicciones pyfunc : {list(pred_cargado)}")
print(f"[diag] Valores reales      : {list(y_test.head(5))}")

## 7. Registro en Unity Catalog

In [ ]:
cliente = MlflowClient()

v_rf = mlflow.register_model(info_rf.model_uri, NOMBRE_MODELO)
cliente.update_registered_model(
    NOMBRE_MODELO,
    description="Clasificador maligno/benigno (breast_cancer de scikit-learn). Práctica del seminario.",
)
cliente.set_model_version_tag(NOMBRE_MODELO, v_rf.version, "test_f1", f"{m_test['f1']:.4f}")
cliente.set_model_version_tag(NOMBRE_MODELO, v_rf.version, "algoritmo", "RandomForest")
cliente.set_registered_model_alias(NOMBRE_MODELO, "champion", v_rf.version)

print(f"[diag] Versión {v_rf.version} registrada y marcada como @champion")

👉 **En la interfaz:** **Catalog › workspace › default › clasificador_cancer_mama**.

### Champion y challenger
El candidato es un gradient boosting con 60 árboles. Ambos se comparan **con el mismo set de prueba**.

In [ ]:
with mlflow.start_run(run_name="gradient_boosting_candidato"):
    candidato = GradientBoostingClassifier(n_estimators=60, random_state=42).fit(X_train, y_train)
    mlflow.log_params({"algoritmo": "GradientBoosting", "n_estimators": 60})
    mlflow.log_metrics({f"test_{k}": v for k, v in metricas(candidato, X_test, y_test).items()})
    mlflow.set_tags({"etapa": "candidato", "autor": "Franco"})
    info_gb = mlflow.sklearn.log_model(
        candidato, name="modelo", input_example=X_train.head(3), skops_trusted_types=TIPOS_CONFIABLES
    )

v_gb = mlflow.register_model(info_gb.model_uri, NOMBRE_MODELO)
cliente.set_model_version_tag(NOMBRE_MODELO, v_gb.version, "algoritmo", "GradientBoosting")
cliente.set_registered_model_alias(NOMBRE_MODELO, "challenger", v_gb.version)


def f1_del_alias(alias: str) -> float:
    modelo = mlflow.pyfunc.load_model(f"models:/{NOMBRE_MODELO}@{alias}")
    return f1_score(y_test, modelo.predict(X_test))


comparacion = pd.DataFrame(
    [
        {"alias": "champion", "version": v_rf.version, "f1_test": f1_del_alias("champion")},
        {"alias": "challenger", "version": v_gb.version, "f1_test": f1_del_alias("challenger")},
    ]
)
display(comparacion)

### Ejercicio 3: regla de promoción
Si el challenger tiene un F1 **estrictamente mayor**, se mueve `@champion` a su versión.
En empate se queda el champion actual: cambiar de modelo tiene costo y sin mejora no se justifica.

In [ ]:
f1_champion = f1_del_alias("champion")
f1_challenger = f1_del_alias("challenger")
v_champion = cliente.get_model_version_by_alias(NOMBRE_MODELO, "champion").version
v_challenger = cliente.get_model_version_by_alias(NOMBRE_MODELO, "challenger").version

print("=== Antes de la decisión ===")
print(f"@champion   -> v{v_champion}  f1 = {f1_champion:.4f}")
print(f"@challenger -> v{v_challenger}  f1 = {f1_challenger:.4f}")
print(f"Diferencia  : {f1_challenger - f1_champion:+.4f}")

if f1_challenger > f1_champion:
    cliente.set_registered_model_alias(NOMBRE_MODELO, "champion", v_challenger)
    decision = f"PROMOVIDO: v{v_challenger} pasa a @champion"
elif f1_challenger == f1_champion:
    decision = f"EMPATE: se mantiene v{v_champion}"
else:
    decision = f"SIN CAMBIO: v{v_champion} sigue siendo mejor"

v_final = cliente.get_model_version_by_alias(NOMBRE_MODELO, "champion").version
print("\n=== Después de la decisión ===")
print(decision)
print(f"@champion apunta ahora a v{v_final}")

## 8. Consumir el modelo en producción
El código de producción apunta al **alias**, nunca a un número de versión.

In [ ]:
version_actual = cliente.get_model_version_by_alias(NOMBRE_MODELO, "champion").version
print(f"[diag] @champion apunta hoy a la versión {version_actual}")

datos_nuevos = spark.createDataFrame(X_test)
predecir = mlflow.pyfunc.spark_udf(spark, f"models:/{NOMBRE_MODELO}@champion", result_type="long")
predicciones = datos_nuevos.withColumn("diagnostico_predicho", predecir(*X_test.columns))
print(f"[diag] Filas con predicción: {predicciones.count()}")
display(predicciones.select("mean radius", "mean texture", "worst area", "diagnostico_predicho").limit(10))

## 9. Tracing
Cada llamada decorada con `@mlflow.trace` deja una traza con entradas, salidas y tiempos.

In [ ]:
@mlflow.trace(span_type="RETRIEVER")
def buscar_muestra(indice: int) -> dict:
    return X_test.iloc[[indice]].to_dict(orient="records")[0]


@mlflow.trace(span_type="TOOL")
def clasificar(muestra: dict) -> int:
    modelo = mlflow.pyfunc.load_model(f"models:/{NOMBRE_MODELO}@champion")
    return int(modelo.predict(pd.DataFrame([muestra]))[0])


@mlflow.trace(span_type="CHAIN")
def responder(indice: int) -> str:
    muestra = buscar_muestra(indice)
    etiqueta = {0: "MALIGNO", 1: "BENIGNO"}[clasificar(muestra)]
    return f"La muestra {indice} se clasifica como {etiqueta} (radio medio {muestra['mean radius']:.2f})."


print(responder(0))

👉 **En la interfaz:** pestaña **Traces** del experimento.

## Conclusiones
1. Separar validación y prueba evita la fuga de información del notebook original: los
   hiperparámetros se eligen sin mirar el set de prueba.
2. En un problema médico el accuracy no basta: se registró el recall de la clase maligna y el AUC.
3. El flujo champion/challenger con alias permite cambiar el modelo de producción sin tocar el
   código que lo consume.

## Limpieza (opcional)

In [ ]:
# MlflowClient().delete_registered_model(NOMBRE_MODELO)